# 09. OOD Detection, Abstention, Conformal Prediction, and Algorithmic Fairness

- **Category:** 11_ai_safety_privacy_and_verification
- **Lineage & Citations:** Chow (1970 *An optimum character recognition system using decision functions*); Hardt, Price & Srebro (2016 *Equality of Opportunity in Supervised Learning*); Geifman & El-Yaniv (2019 *SelectiveNet: A Deep Neural Network with an Integrated Reject Option*); Liu et al. (2020 *Energy-based Out-of-distribution Detection*); Angelopoulos & Bates (2021 *A Gentle Introduction to Conformal Prediction and Distribution-Free Uncertainty Quantification*).
- **Purpose:** Formulate and evaluate safety-critical deployment mechanisms: Helmholtz free energy out-of-distribution (OOD) scoring, split conformal prediction with finite-sample $1-\alpha$ coverage guarantees, SelectiveNet empirical risk and coverage penalties, and Equalized Odds fairness metrics.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [08. AI Watermarking AI Detection and Model Obfuscation](08_ai_watermarking_ai_detection_and_model_obfuscation.ipynb) $\to$ `🔗 Downstream Syntheses:` [10. Cryptographic Verifiable AI ZKML Freivalds and Homomorphic CKKS](10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do not rely on unnormalized Max Softmax Probability (MSP) for OOD detection (causes extreme overconfidence on noise); do not calibrate conformal prediction quantiles on training data (leads to severe test-time under-coverage); do not assume removing a protected attribute prevents disparate impact without audit.

## Mathematical Foundations & Governing Equations

1. **Helmholtz Free Energy**: $E(x) = -T \cdot \ln \sum_{c=1}^C e^{f_c(x) / T}$
2. **Split Conformal Quantile**: $\hat{q} = s_{(k)}, \quad k = \min\left(n, \left\lceil (n + 1)(1 - \alpha) \right\rceil\right) \implies \mathbb{P}(Y \in C(X)) \ge 1 - \alpha$
3. **SelectiveNet Objective**: $\mathcal{L} = \frac{\sum_i \ell_i g_i}{\sum_i g_i + 10^{-8}} + \lambda \left(\max\left(0, c - \frac{1}{B}\sum_i g_i\right)\right)^2$
4. **Equalized Odds Disparity**: $\Delta_{\text{EO}} = |\text{TPR}_0 - \text{TPR}_1| + |\text{FPR}_0 - \text{FPR}_1|$

In [ ]:
import sys, time, math
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/11_ai_safety_privacy_and_verification/09_ood_detection_abstention_conformal_and_algorithmic_fairness.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def energy_ood_score(logits, temperature):
    return -float(temperature) * torch.logsumexp(logits / float(temperature), dim=-1)

def split_conformal_quantile(cal_scores, alpha):
    sorted_scores, _ = torch.sort(cal_scores)
    n = sorted_scores.shape[0]
    idx = min(n - 1, max(0, math.ceil((n + 1) * (1.0 - float(alpha))) - 1))
    return sorted_scores[idx]

def selective_net_loss(ce_losses, selection_probs, target_coverage, penalty_weight):
    phi_hat = torch.mean(selection_probs)
    empirical_risk = torch.sum(ce_losses * selection_probs) / (torch.sum(selection_probs) + 1e-8)
    coverage_deficit = torch.clamp(float(target_coverage) - phi_hat, min=0.0)
    penalty = float(penalty_weight) * (coverage_deficit ** 2)
    return empirical_risk + penalty

def equalized_odds_discrepancy(y_pred, y_true, sensitive_attr):
    m_a0, m_a1 = sensitive_attr == 0.0, sensitive_attr == 1.0
    m_y1, m_y0 = y_true == 1.0, y_true == 0.0
    tpr_0 = torch.sum(y_pred * m_a0 * m_y1) / (torch.sum(m_a0 * m_y1) + 1e-8)
    tpr_1 = torch.sum(y_pred * m_a1 * m_y1) / (torch.sum(m_a1 * m_y1) + 1e-8)
    fpr_0 = torch.sum(y_pred * m_a0 * m_y0) / (torch.sum(m_a0 * m_y0) + 1e-8)
    fpr_1 = torch.sum(y_pred * m_a1 * m_y0) / (torch.sum(m_a1 * m_y0) + 1e-8)
    return torch.abs(tpr_0 - tpr_1) + torch.abs(fpr_0 - fpr_1)

In [ ]:
# Parity verification against golden reference tensors
energy = energy_ood_score(weights["logits"], temperature=1.0)
assert torch.max(torch.abs(energy - weights["energy_ref"])) < 1e-5

q_hat = split_conformal_quantile(weights["cal_scores"], alpha=0.1)
assert torch.max(torch.abs(q_hat - weights["q_hat_ref"])) < 1e-5

sel_loss = selective_net_loss(
    weights["ce_losses"], weights["selection_probs"], target_coverage=0.7, penalty_weight=5.0
)
assert torch.max(torch.abs(sel_loss - weights["selective_loss_ref"])) < 1e-5

eo_disc = equalized_odds_discrepancy(weights["y_pred"], weights["y_true"], weights["sensitive_attr"])
assert torch.max(torch.abs(eo_disc - weights["eo_discrepancy_ref"])) < 1e-5
print("All Energy OOD, Conformal, SelectiveNet, and Fairness parity assertions passed!")

In [ ]:
# Benchmarking safety, abstention, and fairness operators
t0 = time.perf_counter()
for _ in range(500):
    _ = energy_ood_score(weights["logits"], 1.0)
    _ = split_conformal_quantile(weights["cal_scores"], 0.1)
    _ = selective_net_loss(weights["ce_losses"], weights["selection_probs"], 0.7, 5.0)
    _ = equalized_odds_discrepancy(weights["y_pred"], weights["y_true"], weights["sensitive_attr"])
lat_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"Combined OOD, Conformal, SelectiveNet, and Fairness latency: {lat_us:.2f} us")

In [ ]:
# Diagnostic Failure Modes & Auditing
# 1. Energy OOD separation: in-distribution sample 2 has low energy, OOD sample 3 has high energy
for idx, e in enumerate(energy):
    status = "IN-DISTRIBUTION (CONFIDENT)" if e.item() < -4.0 else "OUT-OF-DISTRIBUTION (OOD)"
    print(f"Sample {idx}: Free Energy = {e.item():.2f} -> {status}")
assert energy[2].item() < -8.0, "Sample 2 has massive logits and must have very low energy!"
assert energy[3].item() > -1.0, "Sample 3 has small/diffuse logits and must have higher energy!"

# 2. Conformal Prediction Threshold: guarantees 90% coverage on test distribution
print(f"Calibrated Conformal Quantile threshold (alpha=0.10): q_hat = {q_hat.item():.4f}")
assert q_hat.item() > 0.85, "Top 90% quantile must cover high-confidence nonconformity scores!"

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Energy OOD** detects out-of-distribution inputs by evaluating the unnormalized Helmholtz free energy $E(x) = -T \ln Z(x)$.
2. **Split Conformal Prediction** guarantees finite-sample coverage $\mathbb{P}(Y \in C(X)) \ge 1 - \alpha$ distribution-free.
3. **SelectiveNet** balances accuracy against human review via an explicit selection head and coverage penalty constraint.
4. **Equalized Odds** measures conditional parity across demographic groups, bounding disparity in true and false positive rates.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [09_ood_detection_abstention_conformal_and_algorithmic_fairness.md](../../markdown/11_ai_safety_privacy_and_verification/09_ood_detection_abstention_conformal_and_algorithmic_fairness.md)
- **PyTorch 2.x Module:** [09_ood_detection_abstention_conformal_and_algorithmic_fairness.py](../../pytorch2/11_ai_safety_privacy_and_verification/09_ood_detection_abstention_conformal_and_algorithmic_fairness.py)
- **Functional JAX Module:** [09_ood_detection_abstention_conformal_and_algorithmic_fairness.py](../../jax/11_ai_safety_privacy_and_verification/09_ood_detection_abstention_conformal_and_algorithmic_fairness.py)
- **Native Rust Candle Track:** [09_ood_detection_abstention_conformal_and_algorithmic_fairness.rs](../../candle/11_ai_safety_privacy_and_verification/09_ood_detection_abstention_conformal_and_algorithmic_fairness.rs)
- **Hardware Kernel (CPU / WGSL):** [09_ood_detection_abstention_conformal_and_algorithmic_fairness.rs](../../kernels/11_ai_safety_privacy_and_verification/09_ood_detection_abstention_conformal_and_algorithmic_fairness.rs)
- **Golden SafeTensors:** [09_ood_detection_abstention_conformal_and_algorithmic_fairness.safetensors](../../weights/11_ai_safety_privacy_and_verification/09_ood_detection_abstention_conformal_and_algorithmic_fairness.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 08. AI Watermarking AI Detection and Model Obfuscation](08_ai_watermarking_ai_detection_and_model_obfuscation.ipynb) | [Master Index](../../README.md#11_ai_safety_privacy_and_verification) | [Next: 10. Cryptographic Verifiable AI ZKML Freivalds and Homomorphic CKKS](10_cryptographic_verifiable_ai_zkml_freivalds_and_homomorphic_ckks.ipynb) &rarr;